# ViViT 基线因子：端到端流程

本 Notebook 只负责逐步调用函数；数据、模型、训练和校验逻辑均位于 `vivit_factor` 包中。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from vivit_factor.data import (
    PipelinePaths, build_feature_store, build_history_indices, build_labels,
    build_rolling_splits, build_universe, validate_inputs,
)
from vivit_factor.training import run_parallel_training, smoke_test, validate_outputs

paths = PipelinePaths.default()
OVERWRITE_DATA = False
DEVICE_IDS = (0, 1, 2, 3)
TASKS_PER_GPU = 1
WORKERS_PER_TASK = 8

## 1. 校验 Level2 图片与 Wind 行情输入

In [2]:
validate_inputs(paths)

{'image_count': 1942,
 'first_image_date': '20170103',
 'last_image_date': '20241231',
 'wind_dir': '/root/autodl-tmp/Wind'}

## 2. 构建并检查指定口径的股票池

In [3]:
universe_manifest = build_universe(paths, overwrite=OVERWRITE_DATA)
universe_manifest

universe 2017: 742,307 rows, 727,241 eligible
universe 2018: 817,024 rows, 796,223 eligible
universe 2019: 884,650 rows, 861,690 eligible
universe 2020: 945,918 rows, 916,437 eligible
universe 2021: 1,054,365 rows, 1,027,409 eligible
universe 2022: 1,146,021 rows, 1,120,490 eligible
universe 2023: 1,209,227 rows, 1,194,887 eligible
universe 2024: 1,233,404 rows, 1,200,695 eligible


{'signature': '11d253de641741497eb65b184ec0fc42f8d3a28832d5cdeb901cae3a08315aa3',
 'payload': {'schema_version': 1,
  'image_hash': '5d50ac7039040d16bd79947a1251cd769a9d509080393c0fdb6b6e908833284e',
  'eod': {'name': 'ASHAREEODPRICES.parquet',
   'size': 1312704629,
   'mtime_ns': 1784527001514095592},
  'derivative': {'name': 'ASHAREEODDERIVATIVEINDICATOR.parquet',
   'size': 1828243932,
   'mtime_ns': 1784701006396178144},
  'calendar': {'name': 'ASHARECALENDAR.parquet',
   'size': 194206,
   'mtime_ns': 1784527004450125526},
  'rule': 'valid T image; T and T+1 not suspended; T+1 not limit up/down'},
 'years': ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024'],
 'candidate_rows': 8032916,
 'eligible_rows': 7845072}

## 3. 构建复权 VWAP 标签并检查 MAD 去极值与标准化结果

In [4]:
label_manifest = build_labels(paths, overwrite=OVERWRITE_DATA)
label_sample = pd.read_parquet(paths.labels_dir / 'labels.2018.parquet')
display(label_sample.head())
display(label_sample.groupby('trade_date')['target_zscore'].agg(['count', 'mean', lambda x: x.std(ddof=0)]).head())
label_manifest

labels 2017: 727,222 valid
labels 2018: 796,190 valid
labels 2019: 861,617 valid
labels 2020: 916,332 valid
labels 2021: 1,027,330 valid
labels 2022: 1,120,186 valid
labels 2023: 1,194,694 valid
labels 2024: 1,200,605 valid


,trade_date,instrument_id,wind_code,stock_index,entry_adjusted_vwap,exit_adjusted_vwap,label_end_date,forward_vwap_return_10d,winsorized_forward_return,target_zscore,mad_lower,mad_upper
0,20180102,SH.600000,600000.SH,0,155.086563,160.778534,20180117,0.036702,0.036702,0.948852,-0.185386,0.130575
1,20180102,SH.600004,600004.SH,1,37.096443,39.216026,20180117,0.057137,0.057137,1.278559,-0.185386,0.130575
2,20180102,SH.600006,600006.SH,2,20.818071,20.084021,20180117,-0.035260,-0.035260,-0.212211,-0.185386,0.130575
3,20180102,SH.600007,600007.SH,3,28.529423,28.363708,20180117,-0.005809,-0.005809,0.262973,-0.185386,0.130575
4,20180102,SH.600009,600009.SH,5,132.036407,136.924072,20180117,0.037018,0.037018,0.953945,-0.185386,0.130575


,count,mean,<lambda_0>
trade_date,,,
20180102,3191,-1.583044e-09,1.0
20180103,3204,8.069137e-10,1.0
20180104,3219,-1.613251e-09,1.0
20180105,3209,1.058730e-09,1.0
20180108,3211,4.083781e-09,1.0


{'signature': '7bedd749fc5c90af38c912f5d6def3aafdee822845c3d500b296af8ca631fd81',
 'payload': {'schema_version': 1,
  'universe_signature': '11d253de641741497eb65b184ec0fc42f8d3a28832d5cdeb901cae3a08315aa3',
  'eod': {'name': 'ASHAREEODPRICES.parquet',
   'size': 1312704629,
   'mtime_ns': 1784527001514095592},
  'return': 'adjusted_vwap.shift(-11) / adjusted_vwap.shift(-1) - 1',
  'winsorization': 'median +/- 3 * 1.4826 * MAD',
  'zscore_ddof': 0},
 'years': ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024'],
 'label_rows': 7844176}

## 4. 构建标准化特征与样本索引

In [5]:
feature_manifest = build_feature_store(paths, overwrite=OVERWRITE_DATA)
sample_index = pd.read_parquet(paths.sample_index_path)
display(sample_index.head())
feature_manifest

,sample_index,trade_date,instrument_id,wind_code,stock_index,date_id,t1_trade_date,label_end_date,label_valid
0,0,20170103,SH.600000,600000.SH,0,0,20170104,20170118,True
1,1,20170103,SH.600004,600004.SH,1,0,20170104,20170118,True
2,2,20170103,SH.600005,600005.SH,2,0,20170104,20170118,True
3,3,20170103,SH.600006,600006.SH,3,0,20170104,20170118,True
4,4,20170103,SH.600007,600007.SH,4,0,20170104,20170118,True


{'signature': 'c921b17f5ce21199dbdb1c0b51a28bb34fbf3390644acc64b054d89a28313c5f',
 'payload': {'schema_version': 1,
  'universe_signature': '11d253de641741497eb65b184ec0fc42f8d3a28832d5cdeb901cae3a08315aa3',
  'label_signature': '7bedd749fc5c90af38c912f5d6def3aafdee822845c3d500b296af8ca631fd81',
  'sample_rule': 'is_eligible',
  'normalization': 'per-stock per-channel 8x8 min-max',
  'sample_count': 7845072},
 'sample_count': 7845072}

## 5. 构建 20 日精确历史窗口与滚动样本外切分

In [6]:
build_history_indices(sample_index, paths.history_path)
split_summary = build_rolling_splits(paths)
display(split_summary)

,forecast_year,train_samples,validation_samples,prediction_samples,max_train_label_end,validation_start,max_validation_label_end,forecast_start
0,2018,509039,148855,796223,20171010,20171011,20171229,20180101
1,2019,1123909,324776,861690,20180724,20180725,20181228,20190101
2,2020,1782299,515484,916437,20190517,20190520,20191231,20200101
3,2021,2483533,734299,1027409,20200304,20200305,20201231,20210101
4,2022,3217588,1018080,1120490,20201217,20201218,20211231,20220101
5,2023,4019169,1328941,1194887,20211011,20211012,20221230,20230101
6,2024,4890855,1646802,1200695,20220726,20220727,20231229,20240101
7,2025,5814301,1921883,0,20230515,20230516,20241231,20250101


## 6. 执行单批次前向与反向冒烟测试

In [7]:
smoke_test(paths, forecast_year=2018, seed=42)

{'train_loss': -0.050398048013448715,
 'train_pearson_ic': 0.05677741393446922,
 'validation_ic': -0.13080620656253486,
 'validation_daily_spearman_ic': -0.13080620656253486,
 'validation_daily_pearson_ic': -0.07936633707438821,
 'validation_pooled_pearson_ic': -0.07936633707438821}

## 7. 在多张 GPU 上运行独立的年度 × 随机种子任务

这是耗时最长的步骤；签名一致的 checkpoint 会自动恢复。

In [ ]:
training_summary = run_parallel_training(
    paths, device_ids=DEVICE_IDS, tasks_per_gpu=TASKS_PER_GPU,
    workers=WORKERS_PER_TASK, overwrite=False,
)
display(training_summary)

launch GPU0: 2018/seed=42 pid=33182
launch GPU1: 2018/seed=43 pid=33183
launch GPU2: 2018/seed=44 pid=33184
launch GPU3: 2019/seed=42 pid=33185
done GPU0: 2018/seed=42
done GPU1: 2018/seed=43
done GPU2: 2018/seed=44
factor 2018: 796,223 rows
done GPU3: 2019/seed=42
launch GPU0: 2019/seed=43 pid=33475
launch GPU1: 2019/seed=44 pid=33476
launch GPU2: 2020/seed=42 pid=33477
launch GPU3: 2020/seed=43 pid=33478
done GPU0: 2019/seed=43
done GPU1: 2019/seed=44
factor 2019: 861,690 rows
done GPU2: 2020/seed=42
done GPU3: 2020/seed=43
launch GPU0: 2020/seed=44 pid=33774
launch GPU1: 2021/seed=42 pid=33775
launch GPU2: 2021/seed=43 pid=33776
launch GPU3: 2021/seed=44 pid=33777
done GPU1: 2021/seed=42
done GPU2: 2021/seed=43
launch GPU1: 2022/seed=42 pid=34149
launch GPU2: 2022/seed=43 pid=34150
done GPU0: 2020/seed=44
factor 2020: 916,437 rows
launch GPU0: 2022/seed=44 pid=34547
done GPU3: 2021/seed=44
factor 2021: 1,027,409 rows
launch GPU3: 2023/seed=42 pid=37373


## 8. 校验各年度样本外因子

In [ ]:
validation = validate_outputs(paths)
display(validation)
print(f'因子目录：{paths.factors_dir}')